# 02 · Error analysis: where and why the models fail

Uses the per-frame scores saved by the evaluation (`outputs/evaluation/<model>/scores/<video>.npz`) and the dataset's labels. Figures are saved to `reports/figures/err_*.png`.

Run after the models are evaluated: `jupyter nbconvert --to notebook --execute --inplace notebooks/02_error_analysis.ipynb`.

In [ ]:
import json
import os
import sys
from pathlib import Path
if Path.cwd().name == "notebooks":          # run from the project root, wherever Jupyter started
    os.chdir("..")
sys.path.insert(0, ".")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from src.utils import load_config

cfg = load_config()
FIG = Path("reports/figures"); FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3})
split = json.loads((Path(cfg["paths"]["metadata"]) / "split.json").read_text())

In [ ]:
from sklearn.metrics import roc_auc_score
from src.models import MODELS

def load(model):
    d = Path(f"outputs/evaluation/{model}/scores")
    out = {}
    for f in sorted(d.glob("*.npz")):
        z = np.load(f)
        out[f.stem] = (z["score"].astype(float), z["score_norm"].astype(float), z["labels"].astype(int))
    return out

scores = {m: load(m) for m in MODELS}
scores = {m: s for m, s in scores.items() if s}
SHORT = {m: t.split(":")[0] for m, t in MODELS.items()}       # "Model 1" ...
print("evaluated:", ", ".join(f"{SHORT[m]} ({m})" for m in scores))

## 1. Accuracy per camera
AUC computed inside each camera (all its test frames together). A model can be good on some cameras and close to chance on others.

In [ ]:
def auc(y, s):
    return roc_auc_score(y, s) if 0 < y.sum() < len(y) else np.nan

per_cam = {}
for m, sc in scores.items():
    cams = {}
    for v, (_, nm, y) in sc.items():
        cams.setdefault(v.split("_")[0], []).append((nm, y))
    per_cam[SHORT[m]] = {c: auc(np.concatenate([y for _, y in v]), np.concatenate([s for s, _ in v])) for c, v in cams.items()}
per_cam = pd.DataFrame(per_cam).sort_index()
per_cam.style.format("{:.1%}").background_gradient(cmap="RdYlGn", vmin=0.4, vmax=1.0, axis=None)

In [ ]:
fig, ax = plt.subplots(figsize=(1.6 + 1.1 * per_cam.shape[1], 0.45 * per_cam.shape[0] + 1.2))
im = ax.imshow(per_cam.values, cmap="RdYlGn", vmin=0.4, vmax=1.0, aspect="auto")
ax.set_xticks(range(per_cam.shape[1]), per_cam.columns); ax.set_yticks(range(per_cam.shape[0]), [f"camera {c}" for c in per_cam.index])
for i in range(per_cam.shape[0]):
    for j in range(per_cam.shape[1]):
        if not np.isnan(per_cam.values[i, j]):
            ax.text(j, i, f"{per_cam.values[i, j]:.0%}", ha="center", va="center", fontsize=8)
ax.grid(False); ax.set_title("AUC per camera (per-video normalised scores)")
plt.colorbar(im, ax=ax, fraction=0.03); plt.tight_layout(); plt.savefig(FIG / "err_auc_per_camera.png", bbox_inches="tight"); plt.show()

## 2. Why AUC micro is much lower than AUC per-video
Raw scores of normal frames differ from camera to camera. If the normal frames of one camera score higher than the abnormal frames of another, the pooled (micro) ranking breaks even when every video is ranked well inside.

In [ ]:
best = max(scores, key=lambda m: np.mean([auc(y, s) for s, _, y in scores[m].values() if 0 < y.sum() < len(y)]))
rows = []
for v, (s, _, y) in scores[best].items():
    rows.append({"camera": v.split("_")[0], "normal": np.median(s[y == 0]) if (y == 0).any() else np.nan,
                 "abnormal": np.median(s[y == 1]) if (y == 1).any() else np.nan})
d = pd.DataFrame(rows).groupby("camera").median()
ax = d.plot.bar(figsize=(11, 3.6), color=["#2a78d6", "#eb6834"], width=0.8)
ax.set_title(f"{SHORT[best]} ({best}): median RAW score of normal vs abnormal frames, per camera")
ax.set_ylabel("raw score"); ax.set_xlabel("camera"); plt.tight_layout()
plt.savefig(FIG / "err_raw_score_per_camera.png", bbox_inches="tight"); plt.show()

## 3. The hardest test videos
Per-video AUC of every model; the 10 videos where the models do worst on average.

In [ ]:
pv = pd.DataFrame({SHORT[m]: {v: auc(y, s) for v, (s, _, y) in sc.items()} for m, sc in scores.items()})
pv["mean"] = pv.mean(axis=1)
pv["abnormal share"] = [scores[next(iter(scores))][v][2].mean() for v in pv.index]
pv.sort_values("mean").head(10).style.format("{:.1%}")

## 4. Score over time on a few videos
Normalised score of each model vs. the ground truth (shaded): the easiest, a middle and the hardest video for the models on average.

In [ ]:
order = pv["mean"].dropna().sort_values()
pick = list(dict.fromkeys([order.index[-1], order.index[len(order) // 2], order.index[0]]))   # best, middle, worst
colors = ["#2a78d6", "#eb6834", "#7a5cd6", "#d64f8c", "#1a9e9e", "#8a6a1e"]
fig, axes = plt.subplots(len(pick), 1, figsize=(12, 2.6 * len(pick)), squeeze=False); axes = axes[:, 0]
for ax, v in zip(axes, pick):
    y = scores[next(iter(scores))][v][2]
    ax.fill_between(range(len(y)), 0, y, color="#eb6834", alpha=0.15, step="mid", label="abnormal (truth)")
    for (m, sc), c in zip(scores.items(), colors):
        ax.plot(sc[v][1], color=c, lw=1.2, label=SHORT[m])
    ax.set_title(f"{v}: mean per-video AUC {pv.loc[v, 'mean']:.1%}"); ax.set_ylabel("normalised score"); ax.set_ylim(0, 1.05)
axes[0].legend(ncol=len(scores) + 1, fontsize=8, loc="upper left"); axes[-1].set_xlabel("frame")
plt.tight_layout(); plt.savefig(FIG / "err_score_curves.png", bbox_inches="tight"); plt.show()

## 5. Normal vs abnormal score distributions
The less the two histograms overlap, the better a single threshold separates them.

In [ ]:
n = len(scores); fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 2.8), sharey=True)
axes = np.atleast_1d(axes)
for ax, (m, sc) in zip(axes, scores.items()):
    s = np.concatenate([x[1] for x in sc.values()]); y = np.concatenate([x[2] for x in sc.values()])
    ax.hist(s[y == 0], bins=40, alpha=0.6, density=True, color="#2a78d6", label="normal")
    ax.hist(s[y == 1], bins=40, alpha=0.6, density=True, color="#eb6834", label="abnormal")
    ax.set_title(SHORT[m]); ax.set_xlabel("normalised score")
axes[0].legend(); axes[0].set_ylabel("density")
plt.tight_layout(); plt.savefig(FIG / "err_score_distributions.png", bbox_inches="tight"); plt.show()

## 6. Effect of the temporal smoothing (model 5 vs model 6)
Same network, the only difference is the Gaussian smoothing of the score. Skipped until both are evaluated.

In [ ]:
if {"astt_lite", "astt_lite_smooth"} <= set(scores):
    a, b = scores["astt_lite"], scores["astt_lite_smooth"]
    d = pd.DataFrame({"model 5": {v: auc(y, s) for v, (s, _, y) in a.items()},
                      "model 6": {v: auc(y, s) for v, (s, _, y) in b.items()}}).dropna()
    d["gain"] = d["model 6"] - d["model 5"]
    print(f"smoothing helps on {(d.gain > 0).sum()} of {len(d)} videos; mean gain {d.gain.mean() * 100:+.1f} points")
    ax = d.gain.sort_values().plot.bar(figsize=(12, 3), color=np.where(d.gain.sort_values() > 0, "#1a9e9e", "#d64f8c"))
    ax.set_xticks([]); ax.set_ylabel("AUC model 6 - model 5"); ax.set_title("Per-video gain from smoothing (each bar = one test video)")
    plt.tight_layout(); plt.savefig(FIG / "err_smoothing_gain.png", bbox_inches="tight"); plt.show()
else:
    print("models 5 and 6 are not both evaluated yet")